In [8]:
# master module, put helper function here if needed

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("C:/Users/Carl/Desktop/CSB_project/")
RAW_ROOT = PROJECT_ROOT / "UgandaLSMS"

OUT_ROOT = PROJECT_ROOT / "Finished sections" / "Household"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

WAVE_PATHS = {
    1: RAW_ROOT / "Wave_1",
    2: RAW_ROOT / "Wave_2",
    3: RAW_ROOT / "Wave_3",
    4: RAW_ROOT / "Wave_4",
    5: RAW_ROOT / "Wave_5",
    7: RAW_ROOT / "Wave_7",
    8: RAW_ROOT / "Wave_8",
}

def load_dta(path):
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.strip().str.upper()
    return df


def find_file_case_insensitive(folder, filename):

    filename_upper = filename.upper()

    matches = [p for p in folder.iterdir() if p.name.upper() == filename_upper]

    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {filename} in {folder}")

    if len(matches) > 1:
        raise ValueError(f"Multiple matches for {filename} in {folder}: {matches}")

    return matches[0]


def build_source_variable_map(spec, source):

    source_col = source["source_col"]
    variable_map = {}

    for row in spec["crosswalk_rows"]:
        raw = row.get(source_col)

        if raw is None or str(raw).strip() == "":
            continue

        raw = str(raw).strip().upper()

        if row.get("is_key"):
            target = row["standard_name"].strip().upper()
        else:
            target = row["target"].strip().upper()

        variable_map[raw] = target

    return variable_map


def get_expected_cols(spec):
    expected_cols = []

    for row in spec["crosswalk_rows"]:
        if row.get("is_key"):
            expected_cols.append(row["standard_name"].strip().upper())
        else:
            expected_cols.append(row["target"].strip().upper())

    return list(dict.fromkeys(expected_cols))


def get_metadata_cols(spec):
    metadata_cols = ["WAVE", "SOURCE_FILE", "SOURCE_SECTION"]

    extra_metadata = spec.get("metadata_cols", [])

    for col in extra_metadata:
        col = col.upper()
        if col not in metadata_cols:
            metadata_cols.append(col)

    return metadata_cols


#=========================================
# Section specific helpers

# 1st: GSEC 17. Goal: - One row per HHID + WAVE. H17Q10 = compact month string like "ABCF". H17Q11 = compact reason string, e.g. "ADFJ"


#2nd: GSEC15B Special handling: Wave 7/8 do not have one direct H15BQ12 market price. Instead they have CEB14A, CEB14B, CEB14C. We create artificial H15BQ12 as first non-missing among CEB14A/B/C. We also keep H15BQ12_A/B/C so the split information is not lost. Also empty rows from wave 7/8 are not imported. + itemcode imports from stata become 4 digits, ex: 123_7 -> 1237, helper func for this implemented.

# 3rd: Removes empty rows from dataframe GSEC15CD + same item code fix for codes in wave 3, 7/8 that look like: NNN_N


# 4th: GSEC11 wave7/8 helper. Collects variables from auxiliary files 7_2. Also filters empty income source observations

# 5th: GSEC12, collapse multiple HH observations into one taking first non-missing q1 value observation.

# 6th: GSEC18, recodes transport ID from wave 2/3 from 1,2,3,4 to A,B,C,D. Then only includes rows where q2 or q10 = 1

#7th: GSEC15A: clean and drop wave 7/8 CE01 = 2 (visitors)

#=========================================
#1 = GSEC17
import re
import pandas as pd

def is_yes_value(x):
    if pd.isna(x):
        return False

    s = str(x).strip().upper()

    if s in ["", ".", "NAN", "NONE", "<NA>"]:
        return False

    return s in ["YES", "Y", "1", "1.0", "TRUE"]

MONTH_TO_LETTER = {
    "JANUARY": "A", "JAN": "A", "1": "A", "1.0": "A",
    "FEBRUARY": "B", "FEB": "B", "2": "B", "2.0": "B",
    "MARCH": "C", "MAR": "C", "3": "C", "3.0": "C",
    "APRIL": "D", "APR": "D", "4": "D", "4.0": "D",
    "MAY": "E", "5": "E", "5.0": "E",
    "JUNE": "F", "JUN": "F", "6": "F", "6.0": "F",
    "JULY": "G", "JUL": "G", "7": "G", "7.0": "G",
    "AUGUST": "H", "AUG": "H", "8": "H", "8.0": "H",
    "SEPTEMBER": "I", "SEP": "I", "SEPT": "I", "9": "I", "9.0": "I",
    "OCTOBER": "J", "OCT": "J", "10": "J", "10.0": "J",
    "NOVEMBER": "K", "NOV": "K", "11": "K", "11.0": "K",
    "DECEMBER": "L", "DEC": "L", "12": "L", "12.0": "L",
}

MONTH_ORDER = list("ABCDEFGHIJKL")
REASON_ORDER = list("ABCDEFGHIJKLMN")


def month_to_letter(x):
    if pd.isna(x):
        return None

    s = str(x).strip().upper()

    if s in ["", ".", "NAN", "NONE", "<NA>"]:
        return None

    if s in MONTH_ORDER:
        return s

    m = re.match(r"^([A-L])[\.\):\-\s]+", s)
    if m:
        return m.group(1)

    return MONTH_TO_LETTER.get(s)


def reason_to_letter(x, wave=None):

    if pd.isna(x):
        return None

    s = str(x).strip().upper()

    if s in ["", ".", "NAN", "NONE", "<NA>"]:
        return None

    if s in REASON_ORDER:
        return s

    if s == "X":
        return "N"

    m = re.match(r"^([A-NX])[\.\):\-\s]+", s)
    if m:
        letter = m.group(1)
        return "N" if letter == "X" else letter


    if re.fullmatch(r"\d+(\.0)?", s):
        n = int(float(s))
        if 1 <= n <= 14:
            return REASON_ORDER[n - 1]

    text_map = [
        ("DROUGHT", "A"),
        ("INSECURITY", "B"),
        ("PREVIOUS SEASON", "B"),
        ("PEST", "C"),
        ("PLANT", "D"),
        ("MONEY", "E"),
        ("EXPENSIVE", "F"),
        ("OFFERED FOOD", "G"),
        ("FUEL WOOD", "H"),
        ("DISTRIBUTION", "I"),
        ("BREADWINNER", "J"),
        ("HEAD DIED", "J"),
        ("MOVED AWAY", "J"),
        ("DISTANCE", "K"),
        ("TRANSP", "K"),
        ("MARKET", "L"),
        ("FLOOD", "M"),
        ("WATER LOGGING", "M"),
        ("OTHER", "N"),
    ]

    for pattern, letter in text_map:
        if pattern in s:
            return letter

    return None


def compact_sorted_letters(values, allowed_order):
    letters = [v for v in values if v in allowed_order]
    letters = sorted(set(letters), key=allowed_order.index)

    if len(letters) == 0:
        return pd.NA

    return "".join(letters)


def collapse_long_yes_codes(df_long, hhid_col, code_col, yes_col, converter, allowed_order):

    if df_long is None or len(df_long) == 0:
        return pd.DataFrame(columns=[hhid_col, "COMPACT_CODE"])

    needed = [hhid_col, code_col, yes_col]
    missing = [c for c in needed if c not in df_long.columns]

    if missing:
        raise KeyError(f"Missing columns in auxiliary GSEC17 file: {missing}")

    temp = df_long[needed].copy()

    temp["_YES"] = temp[yes_col].map(is_yes_value)
    temp["_LETTER"] = temp[code_col].map(converter)

    temp = temp[temp["_YES"] & temp["_LETTER"].notna()].copy()

    collapsed = (
        temp
        .groupby(hhid_col, dropna=False)["_LETTER"]
        .apply(lambda x: compact_sorted_letters(x, allowed_order))
        .reset_index(name="COMPACT_CODE")
    )

    return collapsed


def collapse_wide_yes_codes(df, prefix, letters, allowed_order, output_col):

    out = []

    for _, row in df.iterrows():
        selected = []

        for letter in letters:
            source_letter = letter
            standardized_letter = "N" if letter == "X" else letter

            col = f"{prefix}{source_letter}"

            if col in df.columns and is_yes_value(row[col]):
                selected.append(standardized_letter)

        out.append(compact_sorted_letters(selected, allowed_order))

    df[output_col] = out
    return df


def add_gsec17_aux_variables(df_raw, source):

    wave = source["wave"]
    df = df_raw.copy()

    # Waves 2-5: auxiliary long files

    if source.get("q10_file") is not None:
        q10_path = find_file_case_insensitive(WAVE_PATHS[wave], source["q10_file"])
        q10 = load_dta(q10_path)

        q10_collapsed = collapse_long_yes_codes(
            df_long=q10,
            hhid_col="HHID",
            code_col=source["q10_code_col"],
            yes_col=source["q10_yes_col"],
            converter=month_to_letter,
            allowed_order=MONTH_ORDER,
        ).rename(columns={"COMPACT_CODE": "GSEC17_Q10_COMPACT"})

        df = df.merge(q10_collapsed, on="HHID", how="left")

    if source.get("q11_file") is not None:
        q11_path = find_file_case_insensitive(WAVE_PATHS[wave], source["q11_file"])
        q11 = load_dta(q11_path)

        q11_collapsed = collapse_long_yes_codes(
            df_long=q11,
            hhid_col="HHID",
            code_col=source["q11_code_col"],
            yes_col=source["q11_yes_col"],
            converter=lambda x: reason_to_letter(x, wave=wave),
            allowed_order=REASON_ORDER,
        ).rename(columns={"COMPACT_CODE": "GSEC17_Q11_COMPACT"})

        df = df.merge(q11_collapsed, on="HHID", how="left")


    if wave == 8:
        df = collapse_wide_yes_codes(
            df=df,
            prefix="S17Q10",
            letters=list("ABCDEFGHIJKL"),
            allowed_order=MONTH_ORDER,
            output_col="GSEC17_Q10_COMPACT",
        )

        df = collapse_wide_yes_codes(
            df=df,
            prefix="S17Q11",
            letters=list("ABCDEFGHIJKLM") + ["X"],
            allowed_order=REASON_ORDER,
            output_col="GSEC17_Q11_COMPACT",
        )

    return df

#=========================================
# GSEC15B helper

def first_nonmissing_across(row, cols):
    for col in cols:
        if col in row.index and pd.notna(row[col]):
            value = row[col]
            if str(value).strip() not in ["", ".", "nan", "NaN", "None", "<NA>"]:
                return value
    return pd.NA


def fix_wave78_item_code(x):

    if pd.isna(x):
        return pd.NA

    s = str(x).strip()


    if s.endswith(".0"):
        s = s[:-2]

    if "_" in s:
        return s


    if s.isdigit() and len(s) == 4:
        return f"{s[:3]}_{s[3]}"

    return s


def add_gsec15b_market_price(df_raw, source):

    wave = source["wave"]
    df = df_raw.copy()

    if wave in [7, 8]:

        if "CEB01" in df.columns:
            df["CEB01"] = df["CEB01"].map(fix_wave78_item_code)

        price_cols = ["CEB14A", "CEB14B", "CEB14C"]

        for col in price_cols:
            if col not in df.columns:
                df[col] = pd.NA

        activity_cols = [
            "CEB04",
            "CEB06",
            "CEB07",
            "CEB08",
            "CEB09",
            "CEB10",
            "CEB11",
            "CEB12",
            "CEB13",
            "CEB14A",
            "CEB14B",
            "CEB14C",
            "CEB15",
        ]

        existing_activity_cols = [c for c in activity_cols if c in df.columns]

        if existing_activity_cols:
            temp = df[existing_activity_cols].copy()
            temp = temp.replace(["", " ", ".", "nan", "NaN", "None", "<NA>"], pd.NA)

            keep_mask = temp.notna().any(axis=1)

            before = len(df)
            df = df.loc[keep_mask].copy()
            after = len(df)

            print(f"Wave {wave} GSEC15B: dropped {before - after:,} empty item rows; kept {after:,}")

        df["GSEC15B_Q12_COMBINED"] = df.apply(
            lambda row: first_nonmissing_across(row, price_cols),
            axis=1
        )

        df["GSEC15B_Q12_A"] = df["CEB14A"]
        df["GSEC15B_Q12_B"] = df["CEB14B"]
        df["GSEC15B_Q12_C"] = df["CEB14C"]

    return df

#=========================================
# 3. GSEC15CD empty row removal

def drop_empty_standardized_rows(df, spec):
    key_cols = spec["standard_keys"]
    metadata_cols = get_metadata_cols(spec)

    check_cols = [
        col for col in df.columns
        if col not in metadata_cols + key_cols
    ]

    if not check_cols:
        return df

    temp = df[check_cols].copy()

    temp = temp.replace(["", " ", ".", "nan", "NaN", "None", "<NA>"], pd.NA)

    keep_mask = temp.notna().any(axis=1)

    before = len(df)
    df = df.loc[keep_mask].copy()
    after = len(df)

    print(f"Dropped {before - after:,} empty standardized rows; kept {after:,}")

    return df

def fix_wave78_item_code(x):

    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s.endswith(".0"):
        s = s[:-2]

    if "_" in s:
        return s

    if s.isdigit() and len(s) == 4:
        return f"{s[:3]}_{s[3]}"

    return s


def fix_gsec15cd_wave78_item_codes(df_raw, source):

    wave = source["wave"]
    source_section = source["source_section"].upper()

    df = df_raw.copy()

    if wave in [7, 8] and source_section == "GSEC15C" and "CEC02" in df.columns:
        df["CEC02"] = df["CEC02"].map(fix_wave78_item_code)

    if wave in [7, 8] and source_section == "GSEC15D" and "CED02" in df.columns:
        df["CED02"] = df["CED02"].map(fix_wave78_item_code)

    if wave == 3 and source_section == "GSEC15D" and "H15DQ2" in df.columns:
        df["H15DQ2"] = df["H15DQ2"].map(fix_wave78_item_code)

    return df

#=========================================
# 4th:

def add_gsec11_q1_from_main(df_raw, source):
    df = df_raw.copy()

    if source.get("q1_file") is not None:
        wave = source["wave"]
        q1_path = find_file_case_insensitive(WAVE_PATHS[wave], source["q1_file"])
        q1 = load_dta(q1_path)

        q1_raw = source["q1_raw"]

        if "HHID" not in q1.columns:
            raise KeyError(f"Missing HHID in {source['q1_file']}")

        if q1_raw not in q1.columns:
            raise KeyError(f"Missing {q1_raw} in {source['q1_file']}")

        q1 = q1[["HHID", q1_raw]].drop_duplicates(subset=["HHID"])
        df = df.merge(q1, on="HHID", how="left")

    return df

def is_yes_gsec11(x):
    if pd.isna(x):
        return False

    s = str(x).strip().upper()

    if s in ["YES", "Y", "1", "1.0", "TRUE"]:
        return True

    return False


def filter_gsec11_income_rows(df):
    q1_col = "H11Q01"
    q4_col = "H11AQ04"

    if q1_col not in df.columns or q4_col not in df.columns:
        return df

    df = df.copy()

    has_q1 = df[q1_col].notna() & ~df[q1_col].astype("string").str.strip().isin(["", ".", "nan", "NaN", "None", "<NA>"])
    is_yes = df[q4_col].map(is_yes_gsec11)

    yes_rows = df[is_yes].copy()

    hhs_with_yes = set(yes_rows["HHID"].astype("string"))
    fallback_source = df[has_q1 & ~df["HHID"].astype("string").isin(hhs_with_yes)].copy()

    fallback_rows = []

    for hhid, group in fallback_source.groupby("HHID", dropna=False):
        row = group.iloc[0].copy()

        for col in ["INCOME_SOURCE", "H11AQ04", "H11AQ05", "H11AQ06"]:
            if col in row.index:
                row[col] = pd.NA

        fallback_rows.append(row)

    if fallback_rows:
        fallback_df = pd.DataFrame(fallback_rows)
        out = pd.concat([yes_rows, fallback_df], ignore_index=True)
    else:
        out = yes_rows

    before = len(df)
    after = len(out)

    print(f"GSEC11 income filter: dropped {before - after:,} non-yes income-source rows; kept {after:,}")

    return out

#=========================================
#5th:

def collapse_gsec12_to_hh(df):
    if "HHID" not in df.columns or "H12Q01" not in df.columns:
        return df

    metadata_cols = [c for c in ["WAVE", "SOURCE_FILE", "SOURCE_SECTION"] if c in df.columns]

    collapsed = (
        df
        .sort_values(metadata_cols + ["HHID"])
        .groupby(metadata_cols + ["HHID"], dropna=False, as_index=False)
        .agg({"H12Q01": lambda x: x.dropna().iloc[0] if x.dropna().shape[0] > 0 else pd.NA})
    )

    before = len(df)
    after = len(collapsed)

    print(f"GSEC12 collapse: reduced {before:,} rows to {after:,} household rows")

    return collapsed

#=========================================
# 6th:

def recode_number_to_letter_series(series, mapping):
    s = series.astype("string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    return s.map(mapping).fillna(s)


def is_one_value(x):
    if pd.isna(x):
        return False

    s = str(x).strip().upper()
    return s in ["1", "1.0", "YES", "Y", "TRUE"]


def clean_gsec18(df):
    df = df.copy()

    q1_map = {"1": "A", "2": "B", "3": "C", "4": "D"}
    q9_map = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E", "6": "F"}

    wave23 = df["WAVE"].isin([2, 3])

    if "H18Q1" in df.columns:
        df["H18Q1"] = df["H18Q1"].astype("string")
        df.loc[wave23, "H18Q1"] = recode_number_to_letter_series(
            df.loc[wave23, "H18Q1"],
            q1_map
        )

    if "H18Q9" in df.columns:
        df["H18Q9"] = df["H18Q9"].astype("string")
        df.loc[wave23, "H18Q9"] = recode_number_to_letter_series(
            df.loc[wave23, "H18Q9"],
            q9_map
        )

    is_sec18 = df["SOURCE_SECTION"].astype("string").str.upper().eq("GSEC18")
    is_sec18b = df["SOURCE_SECTION"].astype("string").str.upper().eq("GSEC18B")

    keep_sec18 = is_sec18 & df["H18Q2"].map(is_one_value)
    keep_sec18b = is_sec18b & df["H18Q10"].map(is_one_value)

    before = len(df)
    df = df.loc[keep_sec18 | keep_sec18b].copy()
    after = len(df)

    print(f"GSEC18 filter: dropped {before - after:,} non-selected transport/activity rows; kept {after:,}")

    return df

#=========================================

def clean_gsec15a(df_raw, source):
    df = df_raw.copy()
    wave = source["wave"]

    if wave in [7, 8] and "CEA01" in df.columns:
        before = len(df)

        df = df[
            pd.to_numeric(df["CEA01"], errors="coerce").eq(1)
        ].copy()

        after = len(df)
        print(f"Wave {wave} GSEC15A: kept household-member row; dropped {before - after:,} visitor rows")

    return df


#=========================================

def standardize_one_source(source, spec):
    wave = source["wave"]
    filename = source["file"]
    source_section = source["source_section"]
    variable_map = source["variable_map"]

    folder = WAVE_PATHS[wave]
    path = find_file_case_insensitive(folder, filename)

    df_raw = load_dta(path)

    if spec["section"] == "GSEC15A":
        df_raw = clean_gsec15a(df_raw, source)

    if spec["section"] == "GSEC17":
        df_raw = add_gsec17_aux_variables(df_raw, source)

    if spec["section"] == "GSEC15B":
        df_raw = add_gsec15b_market_price(df_raw, source)

    if spec["section"] == "GSEC15CD":
        df_raw = fix_gsec15cd_wave78_item_codes(df_raw, source)

    if spec["section"] == "GSEC11":
        df_raw = add_gsec11_q1_from_main(df_raw, source)

    available_map = {
        raw: target
        for raw, target in variable_map.items()
        if raw in df_raw.columns
    }

    missing_raw = [
        raw
        for raw in variable_map
        if raw not in df_raw.columns
    ]

    df = df_raw[list(available_map.keys())].copy()
    df = df.rename(columns=available_map)

    duplicated_cols = df.columns[df.columns.duplicated()].tolist()
    if duplicated_cols:
        raise ValueError(
            f"Duplicate standardized columns in wave {wave}, "
            f"{source_section}: {duplicated_cols}"
        )

    expected_cols = get_expected_cols(spec)

    for col in expected_cols:
        if col not in df.columns:
            df[col] = pd.NA

    for key in spec["standard_keys"]:
        if key in df.columns:
            df[key] = df[key].astype("string").str.strip()

    df.insert(0, "SOURCE_SECTION", source_section)
    df.insert(0, "SOURCE_FILE", path.name)
    df.insert(0, "WAVE", wave)

    for meta_col in spec.get("metadata_cols", []):
        meta_col_upper = meta_col.upper()
        source_key = meta_col.lower()

        if source_key in source:
            df[meta_col_upper] = source[source_key]
        elif meta_col in source:
            df[meta_col_upper] = source[meta_col]
        else:
            df[meta_col_upper] = pd.NA

    metadata_cols = get_metadata_cols(spec)
    final_cols = metadata_cols + expected_cols
    df = df[final_cols]

    if spec["section"] == "GSEC18":
        df = clean_gsec18(df)

    if spec["section"] == "GSEC12":
        df = collapse_gsec12_to_hh(df)

    if spec["section"] == "GSEC11":
        df = filter_gsec11_income_rows(df)

    if spec["section"] in ["GSEC15CD", "GSEC15E"]:
        df = drop_empty_standardized_rows(df, spec)

    print(f"\nLoaded Wave {wave}: {path.name}")
    print(f"Source section: {source_section}")
    print(f"Rows: {len(df):,}")
    print(f"Mapped variables found: {len(available_map):,}/{len(variable_map):,}")

    if missing_raw:
        print(f"Missing raw variables: {missing_raw}")

    return df



#=========================================



#=========================================

def stack_section(spec):
    section_name = spec["section"]
    pieces = []

    for source in spec["sources"]:
        try:
            df_source = standardize_one_source(source, spec)
            pieces.append(df_source)
        except FileNotFoundError as e:
            print(f"\nSKIPPED missing file: {e}")

    if not pieces:
        raise ValueError(f"No files loaded for {section_name}")

    df_stacked = pd.concat(pieces, ignore_index=True)

    print(f"\n==============================")
    print(f"STACKED {section_name}")
    print(f"Rows: {len(df_stacked):,}")
    print(f"Columns: {len(df_stacked.columns):,}")
    print("==============================")

    metadata_cols = get_metadata_cols(spec)

    group_cols = [
        c for c in metadata_cols
        if c in df_stacked.columns and c not in ["SOURCE_FILE", "SOURCE_SECTION"]
    ]

    if group_cols:
        print("\nRows by metadata:")
        print(
            df_stacked
            .groupby(group_cols, dropna=False)
            .size()
            .reset_index(name="ROWS")
            .to_string(index=False)
        )

    keys = spec["standard_keys"]
    dup_subset = group_cols + keys

    duplicate_count = df_stacked.duplicated(subset=dup_subset).sum()

    print(f"\nUnique metadata-key rows: {df_stacked[dup_subset].drop_duplicates().shape[0]:,}")
    print(f"Duplicates on metadata-keys: {duplicate_count:,}")

    return df_stacked

#=========================================


def export_section(df, section_name):
    section_name = section_name.upper()

    parquet_path = OUT_ROOT / f"{section_name}_standardized.parquet"
    csv_path = OUT_ROOT / f"{section_name}_standardized.csv"
    excel_path = OUT_ROOT / f"{section_name}_standardized_preview.xlsx"

    df.to_csv(csv_path, index=False)

    preview_n = min(len(df), 10000)
    df.head(preview_n).to_excel(excel_path, index=False)

    df_parquet = df.copy()

    object_cols = df_parquet.select_dtypes(include=["object"]).columns

    for col in object_cols:
        df_parquet[col] = df_parquet[col].astype("string")

    df_parquet.to_parquet(parquet_path, index=False)

    print(f"\nExported:")
    print(f"Parquet: {parquet_path}")
    print(f"CSV:     {csv_path}")
    print(f"Excel preview first {preview_n:,} rows: {excel_path}")

    return parquet_path, csv_path, excel_path




In [2]:
# GSEC10 STANDARDIZATION SPEC

GSEC10_SPEC = {
    "section": "GSEC10",
    "level": "household",
    "standard_keys": ["HHID"],

    "sources": [
        {"wave": 1, "file": "GSEC10A.dta", "source_section": "GSEC10A", "source_col": "W1"},
        {"wave": 2, "file": "GSEC10A.dta", "source_section": "GSEC10A", "source_col": "W2"},
        {"wave": 3, "file": "GSEC10A.dta", "source_section": "GSEC10A", "source_col": "W3"},
        {"wave": 4, "file": "GSEC10_1.dta", "source_section": "GSEC10_1", "source_col": "W4"},
        {"wave": 5, "file": "GSEC10_1.dta", "source_section": "GSEC10_1", "source_col": "W5"},
        {"wave": 7, "file": "GSEC10_1.dta", "source_section": "GSEC10_1", "source_col": "W7"},
        {"wave": 8, "file": "GSEC10_1.dta", "source_section": "GSEC10_1", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HHID",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H10Q1",
            "is_key": False,
            "W1": "H10Q1",
            "W2": "H10Q1",
            "W3": "H10Q1",
            "W4": "H10Q1",
            "W5": "H10Q1",
            "W7": "S10Q01",
            "W8": "S10Q01",
        },
        {
            "target": "H10Q6",
            "is_key": False,
            "W1": "H10Q6",
            "W2": "H10Q6",
            "W3": "H10Q6",
            "W4": "H10Q6",
            "W5": "H10Q6",
            "W7": "S10Q06",
            "W8": "S10Q06",
        },
        {
            "target": "H10Q09",
            "is_key": False,
            "W1": "H10Q09",
            "W2": "H10Q9",
            "W3": "H10Q9",
            "W4": "H10Q9",
            "W5": "H10Q9",
            "W7": "S10Q09",
            "W8": "S10Q09",
        },
    ],
}



In [3]:
#GSEC10 out

for source in GSEC10_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC10_SPEC, source)

gsec10 = stack_section(GSEC10_SPEC)
export_section(gsec10, "GSEC10")

gsec10.head()


Loaded Wave 1: GSEC10A.dta
Source section: GSEC10A
Rows: 2,938
Mapped variables found: 4/4

Loaded Wave 2: GSEC10A.dta
Source section: GSEC10A
Rows: 2,637
Mapped variables found: 4/4

Loaded Wave 3: GSEC10A.dta
Source section: GSEC10A
Rows: 2,833
Mapped variables found: 4/4

Loaded Wave 4: GSEC10_1.dta
Source section: GSEC10_1
Rows: 3,117
Mapped variables found: 4/4

Loaded Wave 5: gsec10_1.dta
Source section: GSEC10_1
Rows: 3,305
Mapped variables found: 4/4

Loaded Wave 7: GSEC10_1.dta
Source section: GSEC10_1
Rows: 3,242
Mapped variables found: 4/4

Loaded Wave 8: GSEC10_1.dta
Source section: GSEC10_1
Rows: 3,066
Mapped variables found: 4/4

STACKED GSEC10
Rows: 21,138
Columns: 7

Rows by metadata:
 WAVE  ROWS
    1  2938
    2  2637
    3  2833
    4  3117
    5  3305
    7  3242
    8  3066

Unique metadata-key rows: 21,138
Duplicates on metadata-keys: 0

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC10_standardized.parquet
CSV:     C:\Users\

,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H10Q1,H10Q6,H10Q09
0,1,GSEC10A.dta,GSEC10A,1013000201,2.0,2.0,8.0
1,1,GSEC10A.dta,GSEC10A,1013000204,2.0,2.0,6.0
2,1,GSEC10A.dta,GSEC10A,1013000206,1.0,2.0,3.0
3,1,GSEC10A.dta,GSEC10A,1013000210,2.0,2.0,NaN
4,1,GSEC10A.dta,GSEC10A,1013000213,2.0,2.0,NaN


In [2]:
 # GSEC9 STANDARDIZATION SPEC

GSEC9_SPEC = {
    "section": "GSEC9",
    "level": "household",
    "standard_keys": ["HHID"],

    "sources": [
        {"wave": 1, "file": "GSEC9.dta", "source_section": "GSEC9", "source_col": "W1"},
        {"wave": 2, "file": "GSEC9A.dta", "source_section": "GSEC9A", "source_col": "W2"},
        {"wave": 3, "file": "GSEC9A.dta", "source_section": "GSEC9A", "source_col": "W3"},
        {"wave": 4, "file": "GSEC9_1.dta", "source_section": "GSEC09_1", "source_col": "W4"},
        {"wave": 5, "file": "GSEC9.dta", "source_section": "GSEC9", "source_col": "W5"},
        {"wave": 7, "file": "GSEC9.dta", "source_section": "GSEC9", "source_col": "W7"},
        {"wave": 8, "file": "GSEC9.dta", "source_section": "GSEC9", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {"target": "HHID", "is_key": True, "standard_name": "HHID",
         "W1": "HHID", "W2": "HHID", "W3": "HHID", "W4": "HHID", "W5": "HHID", "W7": "HHID", "W8": "HHID"},

        {"target": "H9Q1", "is_key": False,
         "W1": "H9Q01", "W2": "H9Q1", "W3": "H9Q1", "W4": "H9Q1", "W5": "H9Q1", "W7": "H9Q01", "W8": "H9Q01"},
        {"target": "H9Q2", "is_key": False,
         "W1": "H9Q02", "W2": "H9Q2", "W3": "H9Q2", "W4": "H9Q2", "W5": "H9Q2", "W7": "H9Q02", "W8": "H9Q02"},
        {"target": "H9Q3", "is_key": False,
         "W1": "H9Q03", "W2": "H9Q3", "W3": "H9Q3", "W4": "H9Q3", "W5": "H9Q3", "W7": "H9Q03", "W8": "H9Q03"},
        {"target": "H9Q4", "is_key": False,
         "W1": "H9Q04", "W2": "H9Q4", "W3": "H9Q4", "W4": "H9Q4", "W5": "H9Q4", "W7": "H9Q04", "W8": "H9Q04"},
        {"target": "H9Q5", "is_key": False,
         "W1": "H9Q05", "W2": "H9Q5", "W3": "H9Q5", "W4": "H9Q5", "W5": "H9Q5", "W7": "H9Q05", "W8": "H9Q05"},
        {"target": "H9Q6", "is_key": False,
         "W1": "H9Q06", "W2": "H9Q6", "W3": "H9Q6", "W4": "H9Q6", "W5": "H9Q6", "W7": "H9Q06", "W8": "H9Q06"},
        {"target": "H9Q7", "is_key": False,
         "W1": "H9Q07", "W2": "H9Q7", "W3": "H9Q7", "W4": "H9Q7", "W5": "H9Q7", "W7": "H9Q07", "W8": "H9Q07"},
        {"target": "H9Q8", "is_key": False,
         "W1": "H9Q08", "W2": "H9Q8", "W3": "H9Q8", "W4": "H9Q8", "W5": "H9Q8", "W7": "H9Q08", "W8": "H9Q08"},
        {"target": "H9Q9A", "is_key": False,
         "W1": "H9Q09A", "W2": "H9Q9A", "W3": "H9Q9A", "W4": "H9Q9A", "W5": "H9Q9A", "W7": "H9Q09A", "W8": "H9Q09A"},
        {"target": "H9Q9B", "is_key": False,
         "W1": "H9Q09B", "W2": "H9Q9B", "W3": "H9Q9B", "W4": "H9Q9B", "W5": "H9Q9B", "W7": "H9Q09B", "W8": "H9Q09B"},
        {"target": "H9Q10", "is_key": False,
         "W1": "H9Q10", "W2": "H9Q10", "W3": "H9Q10", "W4": "H9Q10", "W5": "H9Q10", "W7": "H9Q10", "W8": "H9Q10"},

        {"target": "H9Q11A", "is_key": False,
         "W1": "H9Q11A", "W2": "H9Q11A", "W3": "H9Q11A", "W4": None, "W5": None, "W7": None, "W8": None},
        {"target": "H9Q11B", "is_key": False,
         "W1": "H9Q11B", "W2": "H9Q11B", "W3": "H9Q11B", "W4": "H9Q11B", "W5": "H9Q11", "W7": "H9Q11", "W8": "H9Q11"},

        {"target": "H9Q12", "is_key": False,
         "W1": "H9Q12", "W2": "H9Q12", "W3": "H9Q12", "W4": "H9Q12", "W5": "H9Q12", "W7": "H9Q12", "W8": "H9Q12"},
        {"target": "H9Q13", "is_key": False,
         "W1": "H9Q13", "W2": "H9Q13", "W3": "H9Q13", "W4": "H9Q13", "W5": "H9Q13", "W7": "H9Q13", "W8": "H9Q13"},
        {"target": "H9Q14", "is_key": False,
         "W1": "H9Q14", "W2": "H9Q14", "W3": "H9Q14", "W4": "H9Q14", "W5": "H9Q14", "W7": "H9Q14", "W8": "H9Q14"},
        {"target": "H9Q16", "is_key": False,
         "W1": "H9Q16", "W2": "H9Q16", "W3": "H9Q16", "W4": "H9Q16", "W5": "H9Q16", "W7": "H9Q16", "W8": "H9Q16"},
        {"target": "H9Q17", "is_key": False,
         "W1": "H9Q17", "W2": "H9Q17", "W3": "H9Q17", "W4": "H9Q17", "W5": "H9Q17", "W7": "H9Q17", "W8": "H9Q17"},

        {"target": "H9Q18", "is_key": False,
         "W1": "H9Q18", "W2": "H9Q18", "W3": "H9Q18", "W4": "H9Q18", "W5": "H9Q18", "W7": None, "W8": None},
        {"target": "H9Q19", "is_key": False,
         "W1": "H9Q19", "W2": "H9Q19", "W3": "H9Q19", "W4": "H9Q19", "W5": "H9Q19", "W7": None, "W8": None},
        {"target": "H9Q20", "is_key": False,
         "W1": "H9Q20", "W2": "H9Q20", "W3": "H9Q20", "W4": "H9Q20", "W5": "H9Q20", "W7": None, "W8": None},
        {"target": "H9Q21", "is_key": False,
         "W1": "H9Q21", "W2": "H9Q21", "W3": "H9Q21", "W4": "H9Q21", "W5": "H9Q21", "W7": None, "W8": None},

        {"target": "H9Q22", "is_key": False,
         "W1": "H9Q22", "W2": "H9Q22", "W3": "H9Q22", "W4": "H9Q22", "W5": "H9Q22", "W7": "H9Q22", "W8": "H9Q22"},
        {"target": "H9Q23", "is_key": False,
         "W1": "H9Q23", "W2": "H9Q23", "W3": "H9Q23", "W4": "H9Q23", "W5": "H9Q23", "W7": "H9Q23", "W8": "H9Q23"},
    ],
}

In [3]:
# GSEC9 out

for source in GSEC9_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC9_SPEC, source)

gsec9 = stack_section(GSEC9_SPEC)
export_section(gsec9, "GSEC9")

gsec9.head()


Loaded Wave 1: GSEC9.dta
Source section: GSEC9
Rows: 2,942
Mapped variables found: 25/25

Loaded Wave 2: GSEC9A.dta
Source section: GSEC9A
Rows: 2,640
Mapped variables found: 25/25

Loaded Wave 3: GSEC9A.dta
Source section: GSEC9A
Rows: 2,835
Mapped variables found: 25/25

Loaded Wave 4: GSEC9_1.dta
Source section: GSEC09_1
Rows: 3,119
Mapped variables found: 24/24

Loaded Wave 5: gsec9.dta
Source section: GSEC9
Rows: 3,312
Mapped variables found: 24/24

Loaded Wave 7: GSEC9.dta
Source section: GSEC9
Rows: 3,242
Mapped variables found: 20/20

Loaded Wave 8: GSEC9.dta
Source section: GSEC9
Rows: 3,078
Mapped variables found: 20/20

STACKED GSEC9
Rows: 21,168
Columns: 28

Rows by metadata:
 WAVE  ROWS
    1  2942
    2  2640
    3  2835
    4  3119
    5  3312
    7  3242
    8  3078

Unique metadata-key rows: 21,168
Duplicates on metadata-keys: 0

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC9_standardized.parquet
CSV:     C:\Users\Carl\Desktop\C

C:\Users\Carl\AppData\Local\Temp\ipykernel_22544\2757489772.py:269: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H9Q1,H9Q2,H9Q3,H9Q4,H9Q5,H9Q6,...,H9Q13,H9Q14,H9Q16,H9Q17,H9Q18,H9Q19,H9Q20,H9Q21,H9Q22,H9Q23
0,1,GSEC9.dta,GSEC9,1013000201,1.0,1.0,2.0,4.0,3.0,1.0,...,NaN,NaN,9.0,2.0,2.0,1.0,3.0,2.0,5.0,1.0
1,1,GSEC9.dta,GSEC9,1013000204,1.0,1.0,1.0,4.0,3.0,1.0,...,NaN,NaN,2.0,2.0,2.0,1.0,2.0,5.0,2.0,1.0
2,1,GSEC9.dta,GSEC9,1013000206,2.0,5.0,1.0,4.0,6.0,3.0,...,1.0,1500.0,9.0,2.0,2.0,1.0,2.0,5.0,4.0,1.0
3,1,GSEC9.dta,GSEC9,1013000210,1.0,1.0,2.0,4.0,3.0,1.0,...,NaN,NaN,1.0,8.0,NaN,NaN,1.0,5.0,2.0,1.0
4,1,GSEC9.dta,GSEC9,1013000213,1.0,1.0,1.0,4.0,96.0,1.0,...,NaN,NaN,1.0,8.0,8.0,1.0,2.0,5.0,8.0,1.0


In [6]:
#GSEC 16 SPEC

GSEC16_SPEC = {
    "section": "GSEC16",
    "level": "household",
    "standard_keys": ["HHID"],

    "sources": [
        {"wave": 1, "file": "GSEC16.dta", "source_section": "GSEC16", "source_col": "W1"},
        {"wave": 2, "file": "GSEC16.dta", "source_section": "GSEC16", "source_col": "W2"},
        {"wave": 3, "file": "GSEC16.dta", "source_section": "GSEC16", "source_col": "W3"},
        {"wave": 4, "file": "GSEC16.dta", "source_section": "GSEC16", "source_col": "W4"},
        {"wave": 5, "file": "GSEC16.dta", "source_section": "GSEC16", "source_col": "W5"},
        {"wave": 7, "file": "GSEC16.dta", "source_section": "GSEC16", "source_col": "W7"},
        {"wave": 8, "file": "GSEC16.dta", "source_section": "GSEC16", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HHID",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H16Q00",
            "is_key": False,
            "W1": "H16Q00",
            "W2": "H16Q00",
            "W3": "H16Q00",
            "W4": "H16Q00",
            "W5": "H16Q00",
            "W7": "S16QA01",
            "W8": "S16QA01",
        },
        {
            "target": "H16Q1",
            "is_key": False,
            "W1": "H16Q01",
            "W2": "H16Q01",
            "W3": "H16Q01",
            "W4": "H16Q01",
            "W5": "H16Q01",
            "W7": "S16Q01",
            "W8": "S16Q01",
        },
        {
            "target": "H16Q2A",
            "is_key": False,
            "W1": "H16Q02A",
            "W2": "H16Q02A",
            "W3": "H16Q02A",
            "W4": "H16Q02A",
            "W5": "H16Q02A",
            "W7": "S16Q02A",
            "W8": "S16Q02A",
        },
        {
            "target": "H16Q2B",
            "is_key": False,
            "W1": "H16Q02B",
            "W2": "H16Q02B",
            "W3": "H16Q02B",
            "W4": "H16Q02B",
            "W5": "H16Q02B",
            "W7": "S16Q02B",
            "W8": "S16Q02B",
        },
        {
            "target": "H16Q3A",
            "is_key": False,
            "W1": "H16Q3A",
            "W2": "H16Q3A",
            "W3": "H16Q3A",
            "W4": "H16Q3A",
            "W5": "H16Q3A",
            "W7": "S10Q03A",
            "W8": "S10Q03A",
        },
        {
            "target": "H16Q3B",
            "is_key": False,
            "W1": "H16Q3B",
            "W2": "H16Q3B",
            "W3": "H16Q3B",
            "W4": "H16Q3B",
            "W5": "H16Q3B",
            "W7": "S16Q03B",
            "W8": "S16Q03B",
        },
        {
            "target": "H16Q3C",
            "is_key": False,
            "W1": "H16Q3C",
            "W2": "H16Q3C",
            "W3": "H16Q3C",
            "W4": "H16Q3C",
            "W5": "H16Q3C",
            "W7": "S16Q03C",
            "W8": "S16Q03C",
        },
        {
            "target": "H16Q3D",
            "is_key": False,
            "W1": "H16Q3D",
            "W2": "H16Q3D",
            "W3": "H16Q3D",
            "W4": "H16Q3D",
            "W5": "H16Q3D",
            "W7": "S16Q03D",
            "W8": "S16Q03D",
        },
        {
            "target": "H16Q4A",
            "is_key": False,
            "W1": "H16Q4A",
            "W2": "H16Q4A",
            "W3": "H16Q4A",
            "W4": "H16Q4A",
            "W5": "H16Q4A",
            "W7": "S16Q04A",
            "W8": "S16Q04A",
        },
        {
            "target": "H16Q4B",
            "is_key": False,
            "W1": "H16Q4B",
            "W2": "H16Q4B",
            "W3": "H16Q4B",
            "W4": "H16Q4B",
            "W5": "H16Q4B",
            "W7": "S16Q04B",
            "W8": "S16Q04B",
        },
        {
            "target": "H16Q4C",
            "is_key": False,
            "W1": "H16Q4C",
            "W2": "H16Q4C",
            "W3": "H16Q4C",
            "W4": "H16Q4C",
            "W5": "H16Q4C",
            "W7": "S16Q04C",
            "W8": "S16Q04C",
        },
    ],
}



In [7]:
#GSEC16 out

for source in GSEC16_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC16_SPEC, source)

gsec16 = stack_section(GSEC16_SPEC)
export_section(gsec16, "GSEC16")

gsec16.head()


Loaded Wave 1: GSEC16.dta
Source section: GSEC16
Rows: 52,813
Mapped variables found: 12/12

Loaded Wave 2: GSEC16.dta
Source section: GSEC16
Rows: 47,752
Mapped variables found: 12/12

Loaded Wave 3: GSEC16.dta
Source section: GSEC16
Rows: 50,549
Mapped variables found: 12/12

Loaded Wave 4: GSEC16.dta
Source section: GSEC16
Rows: 62,133
Mapped variables found: 12/12

Loaded Wave 5: GSEC16.dta
Source section: GSEC16
Rows: 68,560
Mapped variables found: 12/12

Loaded Wave 7: GSEC16.dta
Source section: GSEC16
Rows: 24,100
Mapped variables found: 12/12

Loaded Wave 8: GSEC16.dta
Source section: GSEC16
Rows: 20,920
Mapped variables found: 12/12

STACKED GSEC16
Rows: 326,827
Columns: 15

Rows by metadata:
 WAVE  ROWS
    1 52813
    2 47752
    3 50549
    4 62133
    5 68560
    7 24100
    8 20920

Unique metadata-key rows: 17,196
Duplicates on metadata-keys: 309,631

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC16_standardized.parquet
CSV:     C:

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\2757489772.py:269: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H16Q00,H16Q1,H16Q2A,H16Q2B,H16Q3A,H16Q3B,H16Q3C,H16Q3D,H16Q4A,H16Q4B,H16Q4C
0,1,GSEC16.dta,GSEC16,1013000201,101,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,GSEC16.dta,GSEC16,1013000201,102,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,GSEC16.dta,GSEC16,1013000201,103,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,GSEC16.dta,GSEC16,1013000201,104,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,GSEC16.dta,GSEC16,1013000201,105,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
#GSEC17 SPEc
# ============================================================


GSEC17_SPEC = {
    "section": "GSEC17",
    "level": "household",
    "standard_keys": ["HHID"],

"sources": [
    {
        "wave": 1,
        "file": "GSEC17.dta",
        "source_section": "GSEC17",
        "source_col": "W1",
    },
    {
        "wave": 2,
        "file": "GSEC17A.dta",
        "source_section": "GSEC17A",
        "source_col": "W2",
        "q11_file": "GSEC17B.dta",
        "q11_code_col": "H17Q11B",
        "q11_yes_col": "H17Q11A",
    },
    {
        "wave": 3,
        "file": "GSEC17A.dta",
        "source_section": "GSEC17A",
        "source_col": "W3",
        "q10_file": "GSEC17B.dta",
        "q10_code_col": "H17Q10B",
        "q10_yes_col": "H17Q10A",
        "q11_file": "GSEC17C.dta",
        "q11_code_col": "H17Q11B",
        "q11_yes_col": "H17Q11A",
    },
    {
        "wave": 4,
        "file": "GSEC17_1.dta",
        "source_section": "GSEC17_1",
        "source_col": "W4",
        "q10_file": "GSEC17_2.dta",
        "q10_code_col": "H17Q10B",
        "q10_yes_col": "H17Q10A",
        "q11_file": "GSEC17_3.dta",
        "q11_code_col": "H17Q11B",
        "q11_yes_col": "H17Q11A",
    },
    {
        "wave": 5,
        "file": "GSEC17_1.dta",
        "source_section": "GSEC17_1",
        "source_col": "W5",
        "q10_file": "GSEC17_2.dta",
        "q10_code_col": "H17Q10B",
        "q10_yes_col": "H17Q10A",
        "q11_file": "GSEC17_3.dta",
        "q11_code_col": "H17Q11B",
        "q11_yes_col": "H17Q11A",
    },
    {
        "wave": 8,
        "file": "GSEC17_1.dta",
        "source_section": "GSEC17_1",
        "source_col": "W8",
    },
],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HHID",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H17Q1",
            "is_key": False,
            "W1": "H17Q01",
            "W2": "H17Q01",
            "W3": "H17Q1",
            "W4": "H17Q1",
            "W5": "H17Q1",
            "W8": "S17Q01",
        },
        {
            "target": "H17Q2",
            "is_key": False,
            "W1": "H17Q02",
            "W2": "H17Q02",
            "W3": "H17Q2",
            "W4": "H17Q2",
            "W5": "H17Q2",
            "W8": "S17Q02",
        },
        {
            "target": "H17Q3",
            "is_key": False,
            "W1": "H17Q03",
            "W2": "H17Q03",
            "W3": "H17Q3",
            "W4": "H17Q3",
            "W5": "H17Q3",
            "W8": "S17Q03",
        },
        {
            "target": "H17Q4",
            "is_key": False,
            "W1": "H17Q04",
            "W2": "H17Q04",
            "W3": "H17Q4",
            "W4": "H17Q4",
            "W5": "H17Q4",
            "W8": "S17Q04",
        },
        {
            "target": "H17Q5",
            "is_key": False,
            "W1": "H17Q05",
            "W2": "H17Q05",
            "W3": "H17Q5",
            "W4": "H17Q5",
            "W5": "H17Q5",
            "W8": "S17Q05",
        },
        {
            "target": "H17Q7",
            "is_key": False,
            "W1": "H17Q07",
            "W2": "H17Q07",
            "W3": "H17Q7",
            "W4": "H17Q7",
            "W5": "H17Q7",
            "W8": "S17Q07",
        },
        {
            "target": "H17Q8",
            "is_key": False,
            "W1": "H17Q08",
            "W2": "H17Q08",
            "W3": "H17Q8",
            "W4": "H17Q8",
            "W5": "H17Q8",
            "W8": "S17Q08",
        },
        {
            "target": "H17Q9",
            "is_key": False,
            "W1": "H17Q09",
            "W2": "H17Q09",
            "W3": "H17Q9",
            "W4": "H17Q9",
            "W5": "H17Q9",
            "W8": "S17Q09",
        },
        {
            "target": "H17Q10",
            "is_key": False,
            "W1": "H17Q10",
            "W2": None,
            "W3": "GSEC17_Q10_COMPACT",
            "W4": "GSEC17_Q10_COMPACT",
            "W5": "GSEC17_Q10_COMPACT",
            "W8": "GSEC17_Q10_COMPACT",
        },
        {
            "target": "H17Q11",
            "is_key": False,
            "W1": "H17Q11",
            "W2": "GSEC17_Q11_COMPACT",
            "W3": "GSEC17_Q11_COMPACT",
            "W4": "GSEC17_Q11_COMPACT",
            "W5": "GSEC17_Q11_COMPACT",
            "W8": "GSEC17_Q11_COMPACT",
        },
    ],
}

In [14]:
#GSEC17 out

for source in GSEC17_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC17_SPEC, source)

gsec17 = stack_section(GSEC17_SPEC)
export_section(gsec17, "GSEC17")

gsec17.head()


Loaded Wave 1: GSEC17.dta
Source section: GSEC17
Rows: 2,924
Mapped variables found: 11/11

Loaded Wave 2: GSEC17A.dta
Source section: GSEC17A
Rows: 2,633
Mapped variables found: 10/10

Loaded Wave 3: GSEC17A.dta
Source section: GSEC17A
Rows: 2,831
Mapped variables found: 11/11

Loaded Wave 4: GSEC17_1.dta
Source section: GSEC17_1
Rows: 3,114
Mapped variables found: 11/11

Loaded Wave 5: gsec17_1.dta
Source section: GSEC17_1
Rows: 3,302
Mapped variables found: 11/11

Loaded Wave 8: GSEC17_1.dta
Source section: GSEC17_1
Rows: 3,078
Mapped variables found: 11/11

STACKED GSEC17
Rows: 17,882
Columns: 14

Rows by metadata:
 WAVE  ROWS
    1  2924
    2  2633
    3  2831
    4  3114
    5  3302
    8  3078

Unique metadata-key rows: 17,882
Duplicates on metadata-keys: 0

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC17_standardized.parquet
CSV:     C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC17_standardized.csv
Excel preview firs

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\2039330726.py:494: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H17Q1,H17Q2,H17Q3,H17Q4,H17Q5,H17Q7,H17Q8,H17Q9,H17Q10,H17Q11
0,1,GSEC17.dta,GSEC17,1013000201,1,2.0,2.0,2.0,3.0,NaN,5.0,2.0,,
1,1,GSEC17.dta,GSEC17,1013000204,1,1.0,2.0,2.0,3.0,4.0,4.0,2.0,,
2,1,GSEC17.dta,GSEC17,1013000206,1,1.0,3.0,1.0,3.0,12.0,12.0,2.0,,
3,1,GSEC17.dta,GSEC17,1013000210,1,1.0,3.0,1.0,3.0,12.0,12.0,2.0,,
4,1,GSEC17.dta,GSEC17,1013000213,1,2.0,2.0,2.0,2.0,NaN,NaN,1.0,C,"A,F,M"


In [24]:
#GSEC15B SPEC

GSEC15B_SPEC = {
    "section": "GSEC15B",
    "level": "household_item",
    "standard_keys": ["HHID", "ITEM_CODE"],

    "sources": [
        {"wave": 1, "file": "GSEC15B.dta", "source_section": "GSEC15B", "source_col": "W1"},
        {"wave": 2, "file": "GSEC15B.dta", "source_section": "GSEC15B", "source_col": "W2"},
        {"wave": 3, "file": "GSEC15B.dta", "source_section": "GSEC15B", "source_col": "W3"},
        {"wave": 4, "file": "GSEC15B.dta", "source_section": "GSEC15B", "source_col": "W4"},
        {"wave": 5, "file": "GSEC15B.dta", "source_section": "GSEC15B", "source_col": "W5"},
        {"wave": 7, "file": "GSEC15B.dta", "source_section": "GSEC15B", "source_col": "W7"},
        {"wave": 8, "file": "GSEC15B.dta", "source_section": "GSEC15B", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HH",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H15BQ2",
            "is_key": True,
            "standard_name": "ITEM_CODE",
            "W1": "H15BQ2",
            "W2": "ITMCD",
            "W3": "ITMCD",
            "W4": "ITMCD",
            "W5": "ITMCD",
            "W7": "CEB01",
            "W8": "CEB01",
        },
        {
            "target": "H15BQ3B",
            "is_key": False,
            "W1": "H15BQ3B",
            "W2": "H15BQ3B",
            "W3": "H15BQ3B",
            "W4": "H15BQ3B",
            "W5": "H15BQ3B",
            "W7": "CEB04",
            "W8": "CEB04",
        },
        {
            "target": "H15BQ3C",
            "is_key": False,
            "W1": "H15BQ3C",
            "W2": "UNTCD",
            "W3": "UNTCD",
            "W4": "UNTCD",
            "W5": "UNTCD",
            "W7": "CEB03C",
            "W8": "CEB03C",
        },
        {
            "target": "H15BQ4",
            "is_key": False,
            "W1": "H15BQ4",
            "W2": "H15BQ4",
            "W3": "H15BQ4",
            "W4": "H15BQ4",
            "W5": "H15BQ4",
            "W7": "CEB06",
            "W8": "CEB06",
        },
        {
            "target": "H15BQ5",
            "is_key": False,
            "W1": "H15BQ5",
            "W2": "H15BQ5",
            "W3": "H15BQ5",
            "W4": "H15BQ5",
            "W5": "H15BQ5",
            "W7": "CEB07",
            "W8": "CEB07",
        },
        {
            "target": "H15BQ6",
            "is_key": False,
            "W1": "H15BQ6",
            "W2": "H15BQ6",
            "W3": "H15BQ6",
            "W4": "H15BQ6",
            "W5": "H15BQ6",
            "W7": "CEB08",
            "W8": "CEB08",
        },
        {
            "target": "H15BQ7",
            "is_key": False,
            "W1": "H15BQ7",
            "W2": "H15BQ7",
            "W3": "H15BQ7",
            "W4": "H15BQ7",
            "W5": "H15BQ7",
            "W7": "CEB09",
            "W8": "CEB09",
        },
        {
            "target": "H15BQ8",
            "is_key": False,
            "W1": "H15BQ8",
            "W2": "H15BQ8",
            "W3": "H15BQ8",
            "W4": "H15BQ8",
            "W5": "H15BQ8",
            "W7": "CEB10",
            "W8": "CEB10",
        },
        {
            "target": "H15BQ9",
            "is_key": False,
            "W1": "H15BQ9",
            "W2": "H15BQ9",
            "W3": "H15BQ9",
            "W4": "H15BQ9",
            "W5": "H15BQ9",
            "W7": "CEB11",
            "W8": "CEB11",
        },
        {
            "target": "H15BQ10",
            "is_key": False,
            "W1": "H15BQ10",
            "W2": "H15BQ10",
            "W3": "H15BQ10",
            "W4": "H15BQ10",
            "W5": "H15BQ10",
            "W7": "CEB012",
            "W8": "CEB012",
        },
        {
            "target": "H15BQ11",
            "is_key": False,
            "W1": "H15BQ11",
            "W2": "H15BQ11",
            "W3": "H15BQ11",
            "W4": "H15BQ11",
            "W5": "H15BQ11",
            "W7": "CEB013",
            "W8": "CEB013",
        },
        {
            "target": "H15BQ12",
            "is_key": False,
            "W1": "H15BQ12",
            "W2": "H15BQ12",
            "W3": "H15BQ12",
            "W4": "H15BQ12",
            "W5": "H15BQ12",
            "W7": "GSEC15B_Q12_COMBINED",
            "W8": "GSEC15B_Q12_COMBINED",
        },
        {
            "target": "H15BQ13",
            "is_key": False,
            "W1": "H15BQ13",
            "W2": "H15BQ13",
            "W3": "H15BQ13",
            "W4": "H15BQ13",
            "W5": "H15BQ13",
            "W7": "CEB15",
            "W8": "CEB15",
        },
        {
            "target": "H15BQ12_A",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": "GSEC15B_Q12_A",
            "W8": "GSEC15B_Q12_A",
        },
        {
            "target": "H15BQ12_B",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": "GSEC15B_Q12_B",
            "W8": "GSEC15B_Q12_B",
        },
        {
            "target": "H15BQ12_C",
            "is_key": False,
            "W1": None,
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": "GSEC15B_Q12_C",
            "W8": "GSEC15B_Q12_C",
        },
    ],
}

In [25]:
#GSEC15B out

for source in GSEC15B_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC15B_SPEC, source)

gsec15b = stack_section(GSEC15B_SPEC)
export_section(gsec15b, "GSEC15B")

gsec15b.head()


Loaded Wave 1: GSEC15B.dta
Source section: GSEC15B
Rows: 38,394
Mapped variables found: 14/14

Loaded Wave 2: GSEC15b.dta
Source section: GSEC15B
Rows: 35,232
Mapped variables found: 14/14

Loaded Wave 3: GSEC15B.dta
Source section: GSEC15B
Rows: 38,743
Mapped variables found: 14/14

Loaded Wave 4: GSEC15B.dta
Source section: GSEC15B
Rows: 50,422
Mapped variables found: 14/14

Loaded Wave 5: gsec15b.dta
Source section: GSEC15B
Rows: 49,751
Mapped variables found: 14/14
Wave 7 GSEC15B: dropped 362,321 empty item rows; kept 53,211

Loaded Wave 7: GSEC15B.dta
Source section: GSEC15B
Rows: 53,211
Mapped variables found: 17/17
Wave 8 GSEC15B: dropped 336,859 empty item rows; kept 48,575

Loaded Wave 8: GSEC15B.dta
Source section: GSEC15B
Rows: 48,575
Mapped variables found: 17/17

STACKED GSEC15B
Rows: 314,328
Columns: 20

Rows by metadata:
 WAVE  ROWS
    1 38394
    2 35232
    3 38743
    4 50422
    5 49751
    7 53211
    8 48575

Unique metadata-key rows: 314,317
Duplicates on metada

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\2690117423.py:571: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns



Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15B_standardized.parquet
CSV:     C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15B_standardized.csv
Excel preview first 10,000 rows: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15B_standardized_preview.xlsx


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,ITEM_CODE,H15BQ3B,H15BQ3C,H15BQ4,H15BQ5,H15BQ6,H15BQ7,H15BQ8,H15BQ9,H15BQ10,H15BQ11,H15BQ12,H15BQ13,H15BQ12_A,H15BQ12_B,H15BQ12_C
0,1,GSEC15B.dta,GSEC15B,1013000201,110.0,3.0,1.0,2.5,6250.0,NaN,NaN,NaN,NaN,NaN,NaN,2500.0,NaN,<NA>,<NA>,<NA>
1,1,GSEC15B.dta,GSEC15B,1013000201,127.0,1.0,4.0,3.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,<NA>,<NA>,<NA>
2,1,GSEC15B.dta,GSEC15B,1013000201,114.0,2.0,41.0,1.0,2500.0,NaN,NaN,NaN,NaN,NaN,NaN,2500.0,NaN,<NA>,<NA>,<NA>
3,1,GSEC15B.dta,GSEC15B,1013000201,148.0,3.0,53.0,1.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,<NA>,<NA>,<NA>
4,1,GSEC15B.dta,GSEC15B,1013000201,147.0,3.0,1.0,1.0,2600.0,NaN,NaN,NaN,NaN,NaN,NaN,2600.0,NaN,<NA>,<NA>,<NA>


In [28]:
# GSEC15BB SPEC AND OUT
# Notes:
# - H15BQ14 and H15BQ15 are available across waves, in wave 7 & 8 sugar and maizeflour is missing and replaced with cooking fat. Here I used "sample taken" as "HH consumed this item" dummy
# H15BQ16B-H15BQ19B are only kept from Wave 1.


GSEC15BB_SPEC = {
    "section": "GSEC15BB",
    "level": "household_item",
    "standard_keys": ["HHID", "ITEM_CODE"],

    "sources": [
        {"wave": 1, "file": "GSEC15BB.dta", "source_section": "GSEC15BB", "source_col": "W1"},
        {"wave": 2, "file": "GSEC15BB.dta", "source_section": "GSEC15BB", "source_col": "W2"},
        {"wave": 3, "file": "GSEC15BB.dta", "source_section": "GSEC15BB", "source_col": "W3"},
        {"wave": 4, "file": "GSEC15BB.dta", "source_section": "GSEC15BB", "source_col": "W4"},
        {"wave": 5, "file": "GSEC15BB.dta", "source_section": "GSEC15BB", "source_col": "W5"},
        {"wave": 7, "file": "GSEC15B_2.dta", "source_section": "GSEC15B_2", "source_col": "W7"},
        {"wave": 8, "file": "GSEC15B_2.dta", "source_section": "GSEC15B_2", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HHID",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H15BQID",
            "is_key": True,
            "standard_name": "ITEM_CODE",
            "W1": "H15BQID",
            "W2": "H15BQID",
            "W3": "H15BQID",
            "W4": "H15BQID",
            "W5": "H15BQID",
            "W7": "FF02",
            "W8": "FF02",
        },
        {
            "target": "H15BQ14",
            "is_key": False,
            "W1": "H15BQ14",
            "W2": "H15BQ14",
            "W3": "H15BQ14",
            "W4": "H15BQ14",
            "W5": "H15BQ14",
            "W7": "FF_21A",
            "W8": "FF_21A",
        },
        {
            "target": "H15BQ15",
            "is_key": False,
            "W1": "H15BQ15",
            "W2": "H15BQ15",
            "W3": "H15BQ15",
            "W4": "H15BQ15",
            "W5": "H15BQ15",
            "W7": None,
            "W8": None,
        },
        {
            "target": "H15BQ16B",
            "is_key": False,
            "W1": "H15BQ16B",
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "H15BQ17B",
            "is_key": False,
            "W1": "H15BQ17B",
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "H15BQ18B",
            "is_key": False,
            "W1": "H15BQ18B",
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
        {
            "target": "H15BQ19B",
            "is_key": False,
            "W1": "H15BQ19B",
            "W2": None,
            "W3": None,
            "W4": None,
            "W5": None,
            "W7": None,
            "W8": None,
        },
    ],
}

for source in GSEC15BB_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC15BB_SPEC, source)

gsec15bb = stack_section(GSEC15BB_SPEC)
export_section(gsec15bb, "GSEC15BB")

gsec15bb.head()


Loaded Wave 1: GSEC15BB.dta
Source section: GSEC15BB
Rows: 11,737
Mapped variables found: 8/8

Loaded Wave 2: GSEC15BB.dta
Source section: GSEC15BB
Rows: 10,584
Mapped variables found: 4/4

Loaded Wave 3: GSEC15BB.dta
Source section: GSEC15BB
Rows: 11,112
Mapped variables found: 4/4

Loaded Wave 4: GSEC15BB.dta
Source section: GSEC15BB
Rows: 15,594
Mapped variables found: 4/4

Loaded Wave 5: gsec15bb.dta
Source section: GSEC15BB
Rows: 16,525
Mapped variables found: 4/4

Loaded Wave 7: GSEC15B_2.dta
Source section: GSEC15B_2
Rows: 9,726
Mapped variables found: 3/3

Loaded Wave 8: GSEC15B_2.dta
Source section: GSEC15B_2
Rows: 5,754
Mapped variables found: 3/3

STACKED GSEC15BB
Rows: 81,032
Columns: 11

Rows by metadata:
 WAVE  ROWS
    1 11737
    2 10584
    3 11112
    4 15594
    5 16525
    7  9726
    8  5754

Unique metadata-key rows: 81,032
Duplicates on metadata-keys: 0

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15BB_standardized.parque

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\2690117423.py:571: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,ITEM_CODE,H15BQ14,H15BQ15,H15BQ16B,H15BQ17B,H15BQ18B,H15BQ19B
0,1,GSEC15BB.dta,GSEC15BB,1013000201,113.0,2.0,NaN,NaN,NaN,NaN,NaN
1,1,GSEC15BB.dta,GSEC15BB,1013000201,127.0,1.0,3.0,NaN,5.0,NaN,NaN
2,1,GSEC15BB.dta,GSEC15BB,1013000201,147.0,1.0,3.0,NaN,NaN,2.0,NaN
3,1,GSEC15BB.dta,GSEC15BB,1013000201,150.0,1.0,3.0,NaN,NaN,NaN,2.0
4,1,GSEC15BB.dta,GSEC15BB,1013000204,113.0,1.0,3.0,96.0,NaN,NaN,NaN


In [37]:
# GSEC15C + D SPEC

# Main special mapping:  GSEC15C H15CQ4 = GSEC15D H15DQ5

GSEC15CD_SPEC = {
    "section": "GSEC15CD",
    "level": "household_item",
    "standard_keys": ["HHID", "ITEM_CODE"],

    "sources": [
        {"wave": 1, "file": "GSEC15C.dta", "source_section": "GSEC15C", "source_col": "W1_C"},
        {"wave": 2, "file": "GSEC15C.dta", "source_section": "GSEC15C", "source_col": "W2_C"},
        {"wave": 3, "file": "GSEC15C.dta", "source_section": "GSEC15C", "source_col": "W3_C"},
        {"wave": 4, "file": "GSEC15C.dta", "source_section": "GSEC15C", "source_col": "W4_C"},
        {"wave": 5, "file": "GSEC15C.dta", "source_section": "GSEC15C", "source_col": "W5_C"},
        {"wave": 7, "file": "GSEC15C.dta", "source_section": "GSEC15C", "source_col": "W7_C"},
        {"wave": 8, "file": "GSEC15C.dta", "source_section": "GSEC15C", "source_col": "W8_C"},

        {"wave": 1, "file": "GSEC15D.dta", "source_section": "GSEC15D", "source_col": "W1_D"},
        {"wave": 2, "file": "GSEC15D.dta", "source_section": "GSEC15D", "source_col": "W2_D"},
        {"wave": 3, "file": "GSEC15D.dta", "source_section": "GSEC15D", "source_col": "W3_D"},
        {"wave": 4, "file": "GSEC15D.dta", "source_section": "GSEC15D", "source_col": "W4_D"},
        {"wave": 5, "file": "GSEC15D.dta", "source_section": "GSEC15D", "source_col": "W5_D"},
        {"wave": 7, "file": "GSEC15D.dta", "source_section": "GSEC15D", "source_col": "W7_D"},
        {"wave": 8, "file": "GSEC15D.dta", "source_section": "GSEC15D", "source_col": "W8_D"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1_C": "HHID", "W2_C": "HH", "W3_C": "HHID", "W4_C": "HHID", "W5_C": "HHID", "W7_C": "HHID", "W8_C": "HHID",
            "W1_D": "HHID", "W2_D": "HH", "W3_D": "HHID", "W4_D": "HHID", "W5_D": "HHID", "W7_D": "HHID", "W8_D": "HHID",
        },
        {
            "target": "H15CQ2",
            "is_key": True,
            "standard_name": "ITEM_CODE",
            "W1_C": "H15CQ2", "W2_C": "H15CQ2", "W3_C": "H15CQ2", "W4_C": "ITMCD", "W5_C": "ITMCD", "W7_C": "CEC02", "W8_C": "CEC02",
            "W1_D": "H15DQ2", "W2_D": "H15DQ2", "W3_D": "H15DQ2", "W4_D": "H15DQ2", "W5_D": "ITMCD", "W7_D": "CED02", "W8_D": "CED02",
        },
        {
            "target": "H15CQ3",
            "is_key": False,
            "W1_C": "H15CQ3", "W2_C": "H15CQ3", "W3_C": "H15CQ3", "W4_C": "H15CQ3", "W5_C": "H15CQ3", "W7_C": "CEC03", "W8_C": "CEC03",
            "W1_D": None, "W2_D": None, "W3_D": None, "W4_D": None, "W5_D": None, "W7_D": None, "W8_D": None,
        },
        {
            "target": "H15CQ4",
            "is_key": False,
            "W1_C": "H15CQ4", "W2_C": "H15CQ4", "W3_C": "H15CQ4", "W4_C": "H15CQ4", "W5_C": "H15CQ4", "W7_C": "CEC04", "W8_C": "CEC04",
            "W1_D": "H15DQ5", "W2_D": "H15DQ5", "W3_D": "H15DQ5", "W4_D": "H15DQ3", "W5_D": "H15DQ3", "W7_D": "CED03", "W8_D": "CED03",
        },
        {
            "target": "H15CQ5",
            "is_key": False,
            "W1_C": "H15CQ5", "W2_C": "H15CQ5", "W3_C": "H15CQ5", "W4_C": "H15CQ5", "W5_C": "H15CQ5", "W7_C": "CEC05", "W8_C": "CEC05",
            "W1_D": None, "W2_D": None, "W3_D": None, "W4_D": None, "W5_D": None, "W7_D": None, "W8_D": None,
        },
        {
            "target": "H15CQ6",
            "is_key": False,
            "W1_C": "H15CQ6", "W2_C": "H15CQ6", "W3_C": "H15CQ6", "W4_C": "H15CQ6", "W5_C": "H15CQ6", "W7_C": "CEC06", "W8_C": "CEC06",
            "W1_D": None, "W2_D": None, "W3_D": None, "W4_D": None, "W5_D": None, "W7_D": None, "W8_D": None,
        },
        {
            "target": "H15CQ7",
            "is_key": False,
            "W1_C": "H15CQ7", "W2_C": "H15CQ7", "W3_C": "H15CQ7", "W4_C": "H15CQ7", "W5_C": "H15CQ7", "W7_C": "CEC07", "W8_C": "CEC07",
            "W1_D": None, "W2_D": None, "W3_D": None, "W4_D": None, "W5_D": None, "W7_D": None, "W8_D": None,
        },
        {
            "target": "H15CQ8",
            "is_key": False,
            "W1_C": "H15CQ8", "W2_C": "H15CQ8", "W3_C": "H15CQ8", "W4_C": "H15CQ8", "W5_C": "H15CQ8", "W7_C": "CEC08", "W8_C": "CEC08",
            "W1_D": None, "W2_D": None, "W3_D": None, "W4_D": None, "W5_D": None, "W7_D": None, "W8_D": None,
        },
        {
            "target": "H15CQ9",
            "is_key": False,
            "W1_C": "H15CQ9", "W2_C": "H15CQ9", "W3_C": "H15CQ9", "W4_C": "H15CQ9", "W5_C": "H15CQ9", "W7_C": "CEC09", "W8_C": "CEC09",
            "W1_D": None, "W2_D": None, "W3_D": None, "W4_D": None, "W5_D": None, "W7_D": None, "W8_D": None,
        },
        {
            "target": "H15CQ10",
            "is_key": False,
            "W1_C": "H15CQ10", "W2_C": "H15CQ10", "W3_C": "H15CQ10", "W4_C": "H15CQ10", "W5_C": "H15CQ10", "W7_C": "CEC10", "W8_C": "CEC10",
            "W1_D": None, "W2_D": None, "W3_D": None, "W4_D": None, "W5_D": None, "W7_D": None, "W8_D": None,
        },
    ],
}

for source in GSEC15CD_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC15CD_SPEC, source)

gsec15cd = stack_section(GSEC15CD_SPEC)
export_section(gsec15cd, "GSEC15CD")

gsec15cd.head()



Dropped 0 empty standardized rows; kept 29,313

Loaded Wave 1: GSEC15C.dta
Source section: GSEC15C
Rows: 29,313
Mapped variables found: 10/10
Dropped 0 empty standardized rows; kept 24,815

Loaded Wave 2: GSEC15c.dta
Source section: GSEC15C
Rows: 24,815
Mapped variables found: 10/10
Dropped 0 empty standardized rows; kept 26,522

Loaded Wave 3: GSEC15C.dta
Source section: GSEC15C
Rows: 26,522
Mapped variables found: 10/10
Dropped 0 empty standardized rows; kept 34,136

Loaded Wave 4: GSEC15C.dta
Source section: GSEC15C
Rows: 34,136
Mapped variables found: 10/10
Dropped 0 empty standardized rows; kept 35,702

Loaded Wave 5: gsec15c.dta
Source section: GSEC15C
Rows: 35,702
Mapped variables found: 10/10
Dropped 129,845 empty standardized rows; kept 32,131

Loaded Wave 7: GSEC15C.dta
Source section: GSEC15C
Rows: 32,131
Mapped variables found: 10/10
Dropped 159,034 empty standardized rows; kept 37,226

Loaded Wave 8: GSEC15C.dta
Source section: GSEC15C
Rows: 37,226
Mapped variables found: 

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\3619856186.py:647: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns



Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15CD_standardized.parquet
CSV:     C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15CD_standardized.csv
Excel preview first 10,000 rows: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15CD_standardized_preview.xlsx


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,ITEM_CODE,H15CQ3,H15CQ4,H15CQ5,H15CQ6,H15CQ7,H15CQ8,H15CQ9,H15CQ10
0,1,GSEC15C.dta,GSEC15C,1013000201,302,NaN,NaN,NaN,NaN,15000.0,NaN,NaN,NaN
1,1,GSEC15C.dta,GSEC15C,1013000201,308,3.0,4.0,10000.0,NaN,NaN,NaN,NaN,2500.0
2,1,GSEC15C.dta,GSEC15C,1013000201,451,85.0,10.0,1000.0,NaN,NaN,NaN,NaN,100.0
3,1,GSEC15C.dta,GSEC15C,1013000201,452,47.0,5.0,8500.0,NaN,NaN,NaN,NaN,1700.0
4,1,GSEC15C.dta,GSEC15C,1013000201,457,84.0,2.0,2400.0,NaN,NaN,NaN,NaN,1200.0


In [42]:
# GSEC15E SPEC and out

GSEC15E_SPEC = {
    "section": "GSEC15E",
    "level": "household_item",
    "standard_keys": ["HHID", "ITEM_CODE"],

    "sources": [
        {"wave": 1, "file": "GSEC15E.dta", "source_section": "GSEC15E", "source_col": "W1"},
        {"wave": 4, "file": "GSEC15E.dta", "source_section": "GSEC15E", "source_col": "W4"},
        {"wave": 5, "file": "GSEC15E.dta", "source_section": "GSEC15E", "source_col": "W5"},
        {"wave": 7, "file": "GSEC15E.dta", "source_section": "GSEC15E", "source_col": "W7"},
        {"wave": 8, "file": "GSEC15E.dta", "source_section": "GSEC15E", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H15EQ2",
            "is_key": True,
            "standard_name": "ITEM_CODE",
            "W1": "H15EQ2",
            "W4": "H15EQ2",
            "W5": "H15EQ2",
            "W7": "CEE02",
            "W8": "CEE02",
        },
        {
            "target": "H15EQ3",
            "is_key": False,
            "W1": "H15EQ3",
            "W4": "H15EQ3",
            "W5": "H15EQ3",
            "W7": "CEE03",
            "W8": "CEE03",
        },
    ],
}

for source in GSEC15E_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC15E_SPEC, source)

gsec15e = stack_section(GSEC15E_SPEC)
export_section(gsec15e, "GSEC15E")

gsec15e.head()

Dropped 2,053 empty standardized rows; kept 11,675

Loaded Wave 1: GSEC15E.dta
Source section: GSEC15E
Rows: 11,675
Mapped variables found: 3/3
Dropped 1 empty standardized rows; kept 2,897

Loaded Wave 4: GSEC15E.dta
Source section: GSEC15E
Rows: 2,897
Mapped variables found: 3/3
Dropped 29,745 empty standardized rows; kept 0

Loaded Wave 5: gsec15e.dta
Source section: GSEC15E
Rows: 0
Mapped variables found: 3/3
Dropped 16,189 empty standardized rows; kept 3,021

Loaded Wave 7: GSEC15E.dta
Source section: GSEC15E
Rows: 3,021
Mapped variables found: 3/3
Dropped 16,047 empty standardized rows; kept 2,923

Loaded Wave 8: GSEC15E.dta
Source section: GSEC15E
Rows: 2,923
Mapped variables found: 3/3

STACKED GSEC15E
Rows: 20,516
Columns: 6

Rows by metadata:
 WAVE  ROWS
    1 11675
    4  2897
    7  3021
    8  2923

Unique metadata-key rows: 20,516
Duplicates on metadata-keys: 0

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15E_standardized.parquet
C

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\1142903795.py:646: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,ITEM_CODE,H15EQ3
0,1,GSEC15E.dta,GSEC15E,1013000201,801.0,0.0
1,1,GSEC15E.dta,GSEC15E,1013000201,802.0,0.0
2,1,GSEC15E.dta,GSEC15E,1013000201,803.0,0.0
3,1,GSEC15E.dta,GSEC15E,1013000201,804.0,0.0
4,1,GSEC15E.dta,GSEC15E,1013000201,805.0,0.0


In [4]:
# GSEC11 SPEC and out

GSEC11_SPEC = {
    "section": "GSEC11",
    "level": "household_income_source",
    "standard_keys": ["HHID", "H11Q03"],

    "sources": [
        {"wave": 1, "file": "GSEC11.dta", "source_section": "GSEC11", "source_col": "W1"},
        {"wave": 2, "file": "GSEC11.dta", "source_section": "GSEC11", "source_col": "W2"},
        {"wave": 3, "file": "GSEC11.dta", "source_section": "GSEC11", "source_col": "W3"},
        {"wave": 4, "file": "GSEC11A.dta", "source_section": "GSEC11A", "source_col": "W4"},
        {"wave": 5, "file": "GSEC11_2.dta", "source_section": "GSEC11_2", "source_col": "W5", "q1_file": "GSEC11_1.dta", "q1_raw": "H11Q1"},
        {"wave": 7, "file": "GSEC7_2.dta", "source_section": "GSEC7_2", "source_col": "W7", "q1_file": "GSEC7_1.dta", "q1_raw": "S11Q01"},
        {"wave": 8, "file": "GSEC7_2.dta", "source_section": "GSEC7_2", "source_col": "W8", "q1_file": "GSEC7_1.dta", "q1_raw": "S11Q01"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HHID",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H11Q01",
            "is_key": False,
            "W1": "H11Q01",
            "W2": "H11Q1",
            "W3": "H11Q1",
            "W4": "H11Q1",
            "W5": "H11Q1",
            "W7": "S11Q01",
            "W8": "S11Q01",
        },
        {
            "target": "H11Q03",
            "is_key": True,
            "standard_name": "H11Q03",
            "W1": "H11AQ03",
            "W2": "H11Q2",
            "W3": "H11Q2",
            "W4": "H11Q2",
            "W5": "S11Q3",
            "W7": "INCOMESOURCE",
            "W8": "INCOMESOURCE",
        },
        {
            "target": "H11AQ04",
            "is_key": False,
            "W1": "H11AQ04",
            "W2": "H11Q4",
            "W3": "H11Q4",
            "W4": "H11Q4",
            "W5": "S11Q4",
            "W7": "S11Q04",
            "W8": "S11Q04",
        },
        {
            "target": "H11AQ05",
            "is_key": False,
            "W1": "H11AQ05",
            "W2": "H11Q5",
            "W3": "H11Q5",
            "W4": "H11Q5",
            "W5": "S11Q5",
            "W7": None,
            "W8": "S11Q05",
        },
        {
            "target": "H11AQ06",
            "is_key": False,
            "W1": "H11AQ06",
            "W2": "H11Q6",
            "W3": "H11Q6",
            "W4": "H11Q6",
            "W5": "S11Q6",
            "W7": None,
            "W8": "S11Q06",
        },
    ],
}

for source in GSEC11_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC11_SPEC, source)

gsec11 = stack_section(GSEC11_SPEC)
export_section(gsec11, "GSEC11")



GSEC11 income filter: dropped 44,173 non-yes income-source rows; kept 5,790

Loaded Wave 1: GSEC11.dta
Source section: GSEC11
Rows: 5,790
Mapped variables found: 6/6
GSEC11 income filter: dropped 39,776 non-yes income-source rows; kept 4,968

Loaded Wave 2: GSEC11.dta
Source section: GSEC11
Rows: 4,968
Mapped variables found: 6/6
GSEC11 income filter: dropped 42,560 non-yes income-source rows; kept 5,079

Loaded Wave 3: GSEC11.dta
Source section: GSEC11
Rows: 5,079
Mapped variables found: 6/6
GSEC11 income filter: dropped 0 non-yes income-source rows; kept 3,118

Loaded Wave 4: GSEC11A.dta
Source section: GSEC11A
Rows: 3,118
Mapped variables found: 6/6
GSEC11 income filter: dropped 42,733 non-yes income-source rows; kept 3,481

Loaded Wave 5: gsec11_2.dta
Source section: GSEC11_2
Rows: 3,481
Mapped variables found: 6/6
GSEC11 income filter: dropped 40,988 non-yes income-source rows; kept 3,434

Loaded Wave 7: GSEC7_2.dta
Source section: GSEC7_2
Rows: 3,434
Mapped variables found: 4/4
G

C:\Users\Carl\AppData\Local\Temp\ipykernel_20544\1462177518.py:817: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


(WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC11_standardized.parquet'),
 WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC11_standardized.csv'),
 WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC11_standardized_preview.xlsx'))

In [54]:
# GSEC12 SPEC

GSEC12_SPEC = {
    "section": "GSEC12",
    "level": "household",
    "standard_keys": ["HHID"],

    "sources": [
        {"wave": 1, "file": "GSEC12.dta", "source_section": "GSEC12", "source_col": "W1"},
        {"wave": 2, "file": "GSEC12.dta", "source_section": "GSEC12", "source_col": "W2"},
        {"wave": 3, "file": "GSEC12.dta", "source_section": "GSEC12", "source_col": "W3"},
        {"wave": 4, "file": "GSEC12.dta", "source_section": "GSEC12", "source_col": "W4"},
        {"wave": 5, "file": "GSEC12_1.dta", "source_section": "GSEC12_1", "source_col": "W5"},
        {"wave": 7, "file": "GSEC12_1.dta", "source_section": "GSEC12_1", "source_col": "W7"},
        {"wave": 8, "file": "GSEC12_1.dta", "source_section": "GSEC12_1", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W2": "HHID",
            "W3": "HHID",
            "W4": "HHID",
            "W5": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H12Q01",
            "is_key": False,
            "W1": "H12Q01",
            "W2": "H12Q1",
            "W3": "H12Q1",
            "W4": "H12Q1",
            "W5": "H12Q1",
            "W7": "NA1A",
            "W8": "NA1A",
        },
    ],
}

for source in GSEC12_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC12_SPEC, source)

gsec12 = stack_section(GSEC12_SPEC)
export_section(gsec12, "GSEC12")

gsec12.head()

GSEC12 collapse: reduced 3,316 rows to 2,917 household rows

Loaded Wave 1: GSEC12.dta
Source section: GSEC12
Rows: 2,917
Mapped variables found: 2/2
GSEC12 collapse: reduced 2,414 rows to 1,619 household rows

Loaded Wave 2: GSEC12.dta
Source section: GSEC12
Rows: 1,619
Mapped variables found: 2/2
GSEC12 collapse: reduced 2,258 rows to 1,577 household rows

Loaded Wave 3: GSEC12.dta
Source section: GSEC12
Rows: 1,577
Mapped variables found: 2/2
GSEC12 collapse: reduced 3,442 rows to 3,119 household rows

Loaded Wave 4: gsec12.dta
Source section: GSEC12
Rows: 3,119
Mapped variables found: 2/2
GSEC12 collapse: reduced 3,305 rows to 3,305 household rows

Loaded Wave 5: gsec12_1.dta
Source section: GSEC12_1
Rows: 3,305
Mapped variables found: 2/2
GSEC12 collapse: reduced 1,738 rows to 1,738 household rows

Loaded Wave 7: GSEC12_1.dta
Source section: GSEC12_1
Rows: 1,738
Mapped variables found: 2/2
GSEC12 collapse: reduced 1,628 rows to 1,628 household rows

Loaded Wave 8: GSEC12_1.dta
Sou

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\2577856568.py:764: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H12Q01
0,1,GSEC12.dta,GSEC12,1013000201,1.0
1,1,GSEC12.dta,GSEC12,1013000204,1.0
2,1,GSEC12.dta,GSEC12,1013000206,1.0
3,1,GSEC12.dta,GSEC12,1013000210,1.0
4,1,GSEC12.dta,GSEC12,1013000213,2.0


In [55]:
# GSEC13 WAVE 1 SPEC
# Due to missing sections throughout waves we only include wave 1 for baseline

GSEC13_WAVE1_SPEC = {
    "section": "GSEC13_WAVE1",
    "level": "household",
    "standard_keys": ["HHID"],

    "sources": [
        {"wave": 1, "file": "GSEC13.dta", "source_section": "GSEC13", "source_col": "W1"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
        },
        {
            "target": "H13Q14",
            "is_key": False,
            "W1": "H13Q14",
        },
        {
            "target": "H13Q15",
            "is_key": False,
            "W1": "H13Q15",
        },
        {
            "target": "H13Q19",
            "is_key": False,
            "W1": "H13Q19",
        },
        {
            "target": "H13Q20",
            "is_key": False,
            "W1": "H13Q20",
        },
        {
            "target": "H13Q25",
            "is_key": False,
            "W1": "H13Q25",
        },
    ],
}

for source in GSEC13_WAVE1_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC13_WAVE1_SPEC, source)

gsec13_wave1 = stack_section(GSEC13_WAVE1_SPEC)
export_section(gsec13_wave1, "GSEC13_wave1")

gsec13_wave1.head()


Loaded Wave 1: GSEC13.dta
Source section: GSEC13
Rows: 2,941
Mapped variables found: 6/6

STACKED GSEC13_WAVE1
Rows: 2,941
Columns: 9

Rows by metadata:
 WAVE  ROWS
    1  2941

Unique metadata-key rows: 2,941
Duplicates on metadata-keys: 0

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC13_WAVE1_standardized.parquet
CSV:     C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC13_WAVE1_standardized.csv
Excel preview first 2,941 rows: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC13_WAVE1_standardized_preview.xlsx


C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\2577856568.py:764: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H13Q14,H13Q15,H13Q19,H13Q20,H13Q25
0,1,GSEC13.dta,GSEC13,1013000201,1.0,NaN,2.0,NaN,2.0
1,1,GSEC13.dta,GSEC13,1013000204,NaN,NaN,2.0,NaN,2.0
2,1,GSEC13.dta,GSEC13,1013000206,NaN,NaN,2.0,NaN,2.0
3,1,GSEC13.dta,GSEC13,1013000210,2.0,1.0,2.0,NaN,2.0
4,1,GSEC13.dta,GSEC13,1013000213,2.0,4.0,2.0,NaN,2.0


In [64]:
# GSEC18 / GSEC18B SPEC

GSEC18_SPEC = {
    "section": "GSEC18",
    "level": "household_road_activity",
    "standard_keys": ["HHID"],

    "sources": [
        {"wave": 1, "file": "GSEC18.dta", "source_section": "GSEC18", "source_col": "W1_18"},
        {"wave": 2, "file": "GSEC18.dta", "source_section": "GSEC18", "source_col": "W2_18"},
        {"wave": 3, "file": "GSEC18.dta", "source_section": "GSEC18", "source_col": "W3_18"},

        {"wave": 1, "file": "GSEC18B.dta", "source_section": "GSEC18B", "source_col": "W1_18B"},
        {"wave": 2, "file": "GSEC18B.dta", "source_section": "GSEC18B", "source_col": "W2_18B"},
        {"wave": 3, "file": "GSEC18B.dta", "source_section": "GSEC18B", "source_col": "W3_18B"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1_18": "HHID", "W2_18": "HHID", "W3_18": "HHID",
            "W1_18B": "HHID", "W2_18B": "HHID", "W3_18B": "HHID",
        },
        {
            "target": "H18Q1",
            "is_key": False,
            "W1_18": "H18Q1", "W2_18": "H18Q1", "W3_18": "H18Q1",
            "W1_18B": None, "W2_18B": None, "W3_18B": None,
        },
        {
            "target": "H18Q2",
            "is_key": False,
            "W1_18": "H18Q2", "W2_18": "H18Q2", "W3_18": "H18Q2",
            "W1_18B": None, "W2_18B": None, "W3_18B": None,
        },
        {
            "target": "H18Q4",
            "is_key": False,
            "W1_18": "H18Q4", "W2_18": "H18Q4", "W3_18": "H18Q4",
            "W1_18B": None, "W2_18B": None, "W3_18B": None,
        },
        {
            "target": "H18Q5",
            "is_key": False,
            "W1_18": "H18Q5", "W2_18": "H18Q5", "W3_18": "H18Q5",
            "W1_18B": None, "W2_18B": None, "W3_18B": None,
        },
        {
            "target": "H18Q9",
            "is_key": False,
            "W1_18": None, "W2_18": None, "W3_18": None,
            "W1_18B": "H18Q9", "W2_18B": "H18Q9", "W3_18B": "H18Q9",
        },
        {
            "target": "H18Q10",
            "is_key": False,
            "W1_18": None, "W2_18": None, "W3_18": None,
            "W1_18B": "H18Q10", "W2_18B": "H18Q10", "W3_18B": "H18Q10",
        },
        {
            "target": "H18Q11",
            "is_key": False,
            "W1_18": None, "W2_18": None, "W3_18": None,
            "W1_18B": "H18Q11", "W2_18B": "H18Q11", "W3_18B": "H18Q11",
        },
    ],
}

for source in GSEC18_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC18_SPEC, source)

gsec18 = stack_section(GSEC18_SPEC)
export_section(gsec18, "GSEC18")

gsec18.head()

GSEC18 filter: dropped 5,620 non-selected transport/activity rows; kept 6,137

Loaded Wave 1: GSEC18.dta
Source section: GSEC18
Rows: 6,137
Mapped variables found: 5/5
GSEC18 filter: dropped 5,459 non-selected transport/activity rows; kept 5,197

Loaded Wave 2: GSEC18.dta
Source section: GSEC18
Rows: 5,197
Mapped variables found: 5/5
GSEC18 filter: dropped 5,519 non-selected transport/activity rows; kept 5,702

Loaded Wave 3: GSEC18.dta
Source section: GSEC18
Rows: 5,702
Mapped variables found: 5/5
GSEC18 filter: dropped 13,872 non-selected transport/activity rows; kept 3,716

Loaded Wave 1: GSEC18B.dta
Source section: GSEC18B
Rows: 3,716
Mapped variables found: 4/4
GSEC18 filter: dropped 12,340 non-selected transport/activity rows; kept 3,570

Loaded Wave 2: GSEC18B.dta
Source section: GSEC18B
Rows: 3,570
Mapped variables found: 4/4
GSEC18 filter: dropped 12,769 non-selected transport/activity rows; kept 4,016

Loaded Wave 3: GSEC18B.dta
Source section: GSEC18B
Rows: 4,016
Mapped vari

C:\Users\Carl\AppData\Local\Temp\ipykernel_3260\1179657530.py:821: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


,WAVE,SOURCE_FILE,SOURCE_SECTION,HHID,H18Q1,H18Q2,H18Q4,H18Q5,H18Q9,H18Q10,H18Q11
0,1,GSEC18.dta,GSEC18,1013000206,C,1.0,5.0,1.0,<NA>,<NA>,<NA>
1,1,GSEC18.dta,GSEC18,1013000206,D,1.0,0.0,2.0,<NA>,<NA>,<NA>
2,1,GSEC18.dta,GSEC18,1013000213,D,1.0,0.0,2.0,<NA>,<NA>,<NA>
3,1,GSEC18.dta,GSEC18,101300021302,B,1.0,0.0,2.0,<NA>,<NA>,<NA>
4,1,GSEC18.dta,GSEC18,101300021302,C,1.0,0.0,2.0,<NA>,<NA>,<NA>


In [9]:
#GSEC15A

GSEC15A_SPEC = {
    "section": "GSEC15A",
    "level": "household",
    "standard_keys": ["HHID"],

    "sources": [
        {"wave": 1, "file": "GSEC15A.dta", "source_section": "GSEC15A", "source_col": "W1"},
        {"wave": 4, "file": "GSEC15.dta", "source_section": "GSEC15", "source_col": "W4"},
        {"wave": 7, "file": "GSEC15A.dta", "source_section": "GSEC15A", "source_col": "W7"},
        {"wave": 8, "file": "GSEC15A.dta", "source_section": "GSEC15A", "source_col": "W8"},
    ],

    "crosswalk_rows": [
        {
            "target": "HHID",
            "is_key": True,
            "standard_name": "HHID",
            "W1": "HHID",
            "W4": "HHID",
            "W7": "HHID",
            "W8": "HHID",
        },
        {
            "target": "H15A1",
            "is_key": False,
            "W1": "H15A1",
            "W4": "T6FQ01A",
            "W7": "CEA01A",
            "W8": "CEA01A",
        },
        {
            "target": "H15A2",
            "is_key": False,
            "W1": "H15A2",
            "W4": "T6FQ01B",
            "W7": "CEA01B",
            "W8": "CEA01B",
        },
        {
            "target": "H15A3",
            "is_key": False,
            "W1": "H15A3",
            "W4": "T6FQ01C",
            "W7": "CEA01C",
            "W8": "CEA01C",
        },
        {
            "target": "H15A4",
            "is_key": False,
            "W1": "H15A4",
            "W4": "T6FQ01D",
            "W7": "CEA01D",
            "W8": "CEA01D",
        },
    ],
}

for source in GSEC15A_SPEC["sources"]:
    source["variable_map"] = build_source_variable_map(GSEC15A_SPEC, source)

gsec15a_raw_waves = stack_section(GSEC15A_SPEC)
export_section(gsec15a_raw_waves, "GSEC15A_raw_waves_1_4_7_8")



Loaded Wave 1: GSEC15A.dta
Source section: GSEC15A
Rows: 2,926
Mapped variables found: 5/5

Loaded Wave 4: GSEC15.dta
Source section: GSEC15
Rows: 3,119
Mapped variables found: 5/5
Wave 7 GSEC15A: kept household-member row; dropped 3,242 visitor rows

Loaded Wave 7: GSEC15A.dta
Source section: GSEC15A
Rows: 3,242
Mapped variables found: 5/5
Wave 8 GSEC15A: kept household-member row; dropped 3,078 visitor rows

Loaded Wave 8: GSEC15A.dta
Source section: GSEC15A
Rows: 3,078
Mapped variables found: 5/5

STACKED GSEC15A
Rows: 12,365
Columns: 8

Rows by metadata:
 WAVE  ROWS
    1  2926
    4  3119
    7  3242
    8  3078

Unique metadata-key rows: 12,365
Duplicates on metadata-keys: 0

Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_RAW_WAVES_1_4_7_8_standardized.parquet
CSV:     C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_RAW_WAVES_1_4_7_8_standardized.csv
Excel preview first 10,000 rows: C:\Users\Carl\Desktop\CSB_project

C:\Users\Carl\AppData\Local\Temp\ipykernel_20544\3773635685.py:840: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


(WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC15A_RAW_WAVES_1_4_7_8_standardized.parquet'),
 WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC15A_RAW_WAVES_1_4_7_8_standardized.csv'),
 WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC15A_RAW_WAVES_1_4_7_8_standardized_preview.xlsx'))

In [12]:
derived_files = {
    2: OUT_ROOT / "GSEC15A_wave2_from_GSEC2.csv",
    3: OUT_ROOT / "GSEC15A_wave3_from_GSEC2.csv",
    5: OUT_ROOT / "GSEC15A_wave5_from_GSEC2.csv",
}

keep_cols = [
    "WAVE",
    "SOURCE_FILE",
    "SOURCE_SECTION",
    "HHID",
    "H15A1",
    "H15A2",
    "H15A3",
    "H15A4",
]

# Important: reset pieces here every time this cell runs
pieces = [gsec15a_raw_waves[keep_cols].copy()]

for wave, path in derived_files.items():
    derived = pd.read_csv(path, dtype=str)

    missing = [c for c in keep_cols if c not in derived.columns]
    if missing:
        raise KeyError(f"{path.name} is missing columns: {missing}")

    # Guard against accidentally loading the wrong file
    print(path.name, derived["WAVE"].value_counts(dropna=False).to_dict())

    if set(derived["WAVE"].astype(str).unique()) != {str(wave)}:
        raise ValueError(f"{path.name} does not only contain wave {wave}")

    pieces.append(derived[keep_cols].copy())

gsec15a_full = pd.concat(pieces, ignore_index=True)

for col in ["H15A1", "H15A2", "H15A3", "H15A4"]:
    gsec15a_full[col] = pd.to_numeric(gsec15a_full[col], errors="coerce")

gsec15a_full["TOTAL_HH_MEMBERS_15A"] = (
    gsec15a_full["H15A1"]
    + gsec15a_full["H15A2"]
    + gsec15a_full["H15A3"]
    + gsec15a_full["H15A4"]
)

check_cols = ["WAVE", "HHID"]

dupes = gsec15a_full[gsec15a_full.duplicated(check_cols, keep=False)]

print("Rows by wave:")
display(gsec15a_full.groupby("WAVE").size().reset_index(name="rows"))

print("Mean derived household size by wave:")
display(
    gsec15a_full
    .groupby("WAVE")["TOTAL_HH_MEMBERS_15A"]
    .mean()
    .reset_index(name="mean_total_hh_members")
)

print("Duplicate Wave-HHID rows:", len(dupes))
display(dupes.sort_values(check_cols).head(20))

print("Rows by wave and source:")
display(
    gsec15a_full
    .groupby(["WAVE", "SOURCE_FILE", "SOURCE_SECTION"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["WAVE", "SOURCE_FILE", "SOURCE_SECTION"])
)

dupes = gsec15a_full[
    gsec15a_full.duplicated(["WAVE", "HHID"], keep=False)
].copy()

display(
    dupes
    .groupby(["WAVE", "SOURCE_FILE", "SOURCE_SECTION"], dropna=False)
    .size()
    .reset_index(name="duplicate_rows")
)

if len(dupes) > 0:
    raise ValueError("GSEC15A has duplicate Wave-HHID rows. Fix before export.")

export_section(gsec15a_full, "GSEC15A_standardized")

GSEC15A_wave2_from_GSEC2.csv {'2': 2714}
GSEC15A_wave3_from_GSEC2.csv {'3': 2850}
GSEC15A_wave5_from_GSEC2.csv {'5': 3305}
Rows by wave:


,WAVE,rows
0,1,2926
1,4,3119
2,7,3242
3,8,3078
4,2,2714
5,3,2850
6,5,3305


Mean derived household size by wave:


,WAVE,mean_total_hh_members
0,1,5.429821
1,4,5.157793
2,7,4.598554
3,8,4.551859
4,2,5.117539
5,3,5.414386
6,5,4.337368


Duplicate Wave-HHID rows: 0


...

,WAVE,SOURCE_FILE,SOURCE_SECTION,rows
0,1,GSEC15A.dta,GSEC15A,2926
1,4,GSEC15.dta,GSEC15,3119
2,7,GSEC15A.dta,GSEC15A,3242
3,8,GSEC15A.dta,GSEC15A,3078
4,2,GSEC2.dta,GSEC2_DERIVED_FOR_GSEC15A,2714
5,3,GSEC2.dta,GSEC2_DERIVED_FOR_GSEC15A,2850
6,5,GSEC2.dta,GSEC2_DERIVED_FOR_GSEC15A,3305


,WAVE,SOURCE_FILE,SOURCE_SECTION,duplicate_rows



Exported:
Parquet: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_STANDARDIZED_standardized.parquet
CSV:     C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_STANDARDIZED_standardized.csv
Excel preview first 10,000 rows: C:\Users\Carl\Desktop\CSB_project\Finished sections\Household\GSEC15A_STANDARDIZED_standardized_preview.xlsx


C:\Users\Carl\AppData\Local\Temp\ipykernel_20544\3773635685.py:840: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_parquet.select_dtypes(include=["object"]).columns


(WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC15A_STANDARDIZED_standardized.parquet'),
 WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC15A_STANDARDIZED_standardized.csv'),
 WindowsPath('C:/Users/Carl/Desktop/CSB_project/Finished sections/Household/GSEC15A_STANDARDIZED_standardized_preview.xlsx'))